# Known 4D-STEM Scan Drift Forward Model

Use this notebook to create a controlled known-drift 4D-STEM dataset from a source master H5. It exports three H5 files:

- `clean0`: clean 0-degree reference crop
- `drift0`: 0-degree scan with known drift
- `drift90`: 90-degree scan with the same known drift

The implementation lives in QuantEM modules. This notebook only sets parameters, launches the forward model, and checks the saved metadata.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO = Path("/home/owner/repos/quantem")
WORKFLOW_DIR = REPO / "notebooks" / "drift" / "dev" / "real"
if str(WORKFLOW_DIR) not in sys.path:
    sys.path.insert(0, str(WORKFLOW_DIR))

from known_scan_drift_runner import (
    KnownScanDriftConfig,
    export_status,
    generate_forward_model_exports,
    print_export_status,
)

## Parameters

Edit only this cell for a new dataset. Drift is `(down_px, right_px)` from the first to last slow-scan line.

In [ ]:
SOURCE_H5 = Path("/home/owner/ssd/data/dasol/20260415_BTOSTO/BTO_18_master.h5")
DRIFT_TOTAL_PX_DOWN_RIGHT = (0.0, 30.0)

DET_BIN = 2
SAVE_CROP = 400
GPU = 0

RUN_FORWARD_MODEL = False
OVERWRITE_EXPORTS = False

config = KnownScanDriftConfig(
    source_h5=SOURCE_H5,
    drift_total_px_down_right=DRIFT_TOTAL_PX_DOWN_RIGHT,
    det_bin=DET_BIN,
    save_crop=SAVE_CROP,
    gpu=GPU,
)
config.summary()

{'source_h5': '/home/owner/ssd/data/dasol/20260415_BTOSTO/BTO_18_master.h5',
 'dataset_label': 'BTO_18',
 'drift_total_px_down_right': (0.0, 30.0),
 'export_dir': '/home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16',
 'output_dir': '/home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30'}

## Run Forward Model

When `RUN_FORWARD_MODEL = True`, this loads the source H5, applies the scan-axis drift model, saves the three exports, writes `entry/quantem/drift` metadata, and saves a vector diagnostic figure plus summary arrays.

In [ ]:
if RUN_FORWARD_MODEL:
    result = generate_forward_model_exports(config, overwrite=OVERWRITE_EXPORTS)
    result
else:
    print("RUN_FORWARD_MODEL is False; not generating exports.")

RUN_FORWARD_MODEL is False; not generating exports.


## Check Export Metadata

The scan crop values are Python slice bounds on the scan axes: `row_start:row_stop`, `col_start:col_stop`. Detector pixels are not cropped or warped here.

In [ ]:
rows = export_status(config)
try:
    import pandas as pd
    display(pd.DataFrame(rows)[[
        "name",
        "exists",
        "known_drift_total_px_down",
        "known_drift_total_px_right",
        "scan_crop_row_start",
        "scan_crop_row_stop",
        "scan_crop_col_start",
        "scan_crop_col_stop",
        "det_bin",
        "detector_shape_px",
        "path",
    ]])
except Exception:
    print_export_status(config)

clean0  missing  /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_ground_truth_crop400_detbin2_master.h5
drift0  missing  /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_down0_right30_image_0_crop400_detbin2_master.h5
drift90 missing  /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_down0_right30_image_1_crop400_detbin2_master.h5


## Forward-Model Diagnostic

The diagnostic figure shows the physical drift vector and the adjusted scan/probe positions used to sample the clean 4D-STEM data. It does not show detector-pixel motion because detector pixels are not rolled or warped.

In [ ]:
if config.forward_vectors_png.exists():
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.imshow(plt.imread(config.forward_vectors_png))
    ax.set_axis_off()
    ax.set_title(config.forward_vectors_png.name)
else:
    print(f"No diagnostic figure yet: {config.forward_vectors_png}")

No diagnostic figure yet: /home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30/BTO_18_known_down0_right30_probe_positions.png


## Outputs

Use these exported masters as input to the locked SSB notebook.

In [ ]:
print("Export directory:", config.export_dir)
for name, path in config.masters.items():
    print(f"{name}: {path}")
print("Summary arrays:", config.forward_summary_npz)

Export directory: /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16
clean0: /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_ground_truth_crop400_detbin2_master.h5
drift0: /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_down0_right30_image_0_crop400_detbin2_master.h5
drift90: /home/owner/ssd/data/dasol/20260415_BTOSTO/quantem/drift/real/BTO_18_known_down0_right30_crop400_detbin2_u16/BTO_18_down0_right30_image_1_crop400_detbin2_master.h5
Summary arrays: /home/owner/repos/quantem/notebooks/drift/dev/outputs/BTO_18_known_down0_right30/BTO_18_known_down0_right30_forward_model_summary.npz
